# Multi-Modal Product Classifier
**Text (descriptions) + Image (product photos) = Category prediction**

- Dataset: Fashion Product Images (Kaggle)

## Kitabxana yükləmək

In [ ]:
!pip install torch torchvision transformers scikit-learn pandas pillow tqdm -q

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

# GPU varsa, yoxsa CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# Hiperparametrlər
IMG_SIZE    = 64
MAX_LEN     = 16
BATCH_SIZE  = 32
EPOCHS      = 3
LR          = 3e-4
MAX_SAMPLES = 5000
TOP_N_CLASSES = 10

## Import and Cleaning Data

In [ ]:
import kagglehub
path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")
CSV_PATH = os.path.join(path, 'styles.csv')
df = pd.read_csv(CSV_PATH, on_bad_lines='skip')
df.head()

In [ ]:
# Mətn sütunu yaratmaq sütunları birlesdirmək
df['text'] = (
    df['gender'].fillna('') + ' ' +
    df['masterCategory'].fillna('') + ' ' +
    df['subCategory'].fillna('') + ' ' +
    df['articleType'].fillna('') + ' ' +
    df['baseColour'].fillna('') + ' ' +
    df['season'].fillna('')
).str.strip()

In [ ]:
# Şəkil yolu
df['img_path'] = df['id'].astype(str).apply(
    lambda x: os.path.join(IMG_DIR, x + '.jpg')
)

In [ ]:
top_classes = df['masterCategory'].value_counts().nlargest(TOP_N_CLASSES).index
df = df[df['masterCategory'].isin(top_classes)].reset_index(drop=True)

In [ ]:
if MAX_SAMPLES:
    df = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=42).reset_index(drop=True)
df.head(2)

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['masterCategory'])
NUM_CLASSES = len(le.classes_)

In [ ]:
NUM_CLASSES

In [ ]:
df.head(3)

## Dataset sinifləndirmək


In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

img_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

class ProductDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Şəkil
        try:
            img = Image.open(row['img_path']).convert('RGB')
        except:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        img = img_transform(img)

        # Mətn
        enc = tokenizer(
            row['text'],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids      = enc['input_ids'].squeeze(0)
        attention_mask = enc['attention_mask'].squeeze(0)

        label = torch.tensor(row['label'], dtype=torch.long)
        return img, input_ids, attention_mask, label

## Train/Val ayırmaq, DataLoader yaratmaq

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_loader = DataLoader(ProductDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(ProductDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## ResNet-18 / DistilBERT / Fusion / Classifier

In [ ]:
class MultiModalModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # ResNet-18
        resnet = models.resnet18(weights='DEFAULT')
        resnet.fc = nn.Identity()           # son layeri sil, feature al
        self.img_encoder = resnet           # çıxış: 512

        # DistilBERT
        self.text_encoder = DistilBertModel.from_pretrained('distilbert-base-uncased')
        # Yalnız son 2 layeri train edir qalanı donur
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        for param in self.text_encoder.transformer.layer[-2:].parameters():
            param.requires_grad = True

        # Fusion  Classifier
        # 512 (resnet) + 768 (distilbert) = 1280
        self.classifier = nn.Sequential(
            nn.Linear(512 + 768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, img, input_ids, attention_mask):
        # Şəkil feature-ları
        img_feat = self.img_encoder(img)

        # CLS token
        text_out = self.text_encoder(input_ids=input_ids,
                                     attention_mask=attention_mask)
        text_feat = text_out.last_hidden_state[:, 0, :]


        combined = torch.cat([img_feat, text_feat], dim=1)
        return self.classifier(combined)


model = MultiModalModel(NUM_CLASSES).to(DEVICE)
print('Model hazırdır!')
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parametr sayı: {total_params:,}')

## Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    ctx = torch.no_grad() if not train else torch.enable_grad()
    with ctx:
        for img, input_ids, attn_mask, labels in tqdm(loader, leave=False):
            img        = img.to(DEVICE)
            input_ids  = input_ids.to(DEVICE)
            attn_mask  = attn_mask.to(DEVICE)
            labels     = labels.to(DEVICE)

            logits = model(img, input_ids, attn_mask)
            loss   = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, macro_f1

print(f'{"Epoch":<8} {"Train Loss":<12} {"Train F1":<12} {"Val Loss":<12} {"Val F1"}')
print('-' * 56)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1 = run_epoch(train_loader, train=True)
    vl_loss, vl_f1 = run_epoch(val_loader,   train=False)
    print(f'{epoch:<8} {tr_loss:<12.4f} {tr_f1:<12.4f} {vl_loss:<12.4f} {vl_f1:.4f}')

## Model Qiymətləndirmə

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for img, input_ids, attn_mask, labels in val_loader:
        logits = model(img.to(DEVICE), input_ids.to(DEVICE), attn_mask.to(DEVICE))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(
    all_labels, all_preds,
    target_names=le.classes_,
    zero_division=0
))

macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
print(macro_f1)

In [ ]:
# Modeli yadda saxlaamaq
torch.save(model.state_dict(), 'multimodal_model.pth')